# MR 데이터 전처리 — 원본부터 하나하나 (노출·결과 만들기)

**목적**: MR을 돌리려면 원본(eQTL·GWAS)을 노출(exposure)·결과(outcome) 표로 가공해야 함. 이 노트북은 원본 파일을 직접 읽어 전처리 과정을 단계별로 보여줌.

**흐름**: 원본 eQTL VCF → 1 파싱 → 2 도구변수 선택(Supp8) → 3 결과 GWAS 매칭 → 4 harmonise → MR 입력 완성

> 예시 = FN1. 실제 R(TwoSampleMR) 전처리를 Python으로 재현.

## STEP 1. 원본 eQTL VCF 열어보기

eQTL = 'SNP이 유전자 발현을 얼마나 바꾸나' 데이터. 원본은 GWAS-VCF 형식.
- FORMAT = ES:SE:LP:AF:SS → ES=beta(발현효과), SE, LP=-log10(p), AF=빈도, SS=표본수

In [ ]:
raw = read_vcf("2-1. eqtl-a-ENSG00000115414.vcf")   # 원본 eQTL
raw.head()

FN1 eQTL VCF 총 SNP: 18,851개


chr,pos,SNP,REF,ALT,FORMAT,값(ES:SE:LP:AF:SS:ID)
1,1247494,rs12103,T,C,ES:SE:LP:AF:SS:ID,-0.00198678:0.0138355:0.0525789:0.754845:18440:rs12103
1,1894284,rs4648739,T,C,ES:SE:LP:AF:SS:ID,0.00106262:0.0141683:0.026773:0.228823:14882:rs4648739
1,2069172,rs425277,C,T,ES:SE:LP:AF:SS:ID,-0.0357589:0.0132093:2.16843:0.282999:27015:rs425277
1,2146165,rs78265569,C,A,ES:SE:LP:AF:SS:ID,0.0230602:0.0228523:0.504491:0.073182:31155:rs78265569
1,2233961,rs11576356,G,A,ES:SE:LP:AF:SS:ID,-0.00178623:0.0168512:0.0383779:0.146087:23874:rs11576356


→ 원본은 **18,851개 SNP** (FN1 근처 전부). 마지막 열에 ES:SE:LP:AF:SS가 콜론으로 붙어있음 → 풀어야 함.

## STEP 2. FORMAT 파싱 → 노출(exposure) 후보 표

콜론으로 붙은 ES:SE:LP:AF:SS를 컬럼으로 분리. beta=ES, pval=10^(-LP), eaf=AF.

In [ ]:
p = raw["값"].str.split(":", expand=True)   # ES:SE:LP:AF:SS 분리
beta=p[0]; se=p[1]; pval=10**(-p[2].astype(float)); eaf=p[3]

SNP,effect_allele,other_allele,beta_exposure,se,pval,eaf,N
rs12103,C,T,-0.0020,0.0138,8.86e-01,0.755,18440
rs4648739,C,T,0.0011,0.0142,9.40e-01,0.229,14882
rs425277,T,C,-0.0358,0.0132,6.79e-03,0.283,27015
rs78265569,A,C,0.0231,0.0229,3.13e-01,0.073,31155
rs11576356,A,G,-0.0018,0.0169,9.15e-01,0.146,23874


→ SNP마다 beta(발현효과)·se·pval·eaf 정리됨. 하지만 18,851개 전부 안 씀 → 다음 단계 선별.

## STEP 3. 도구변수 선택 — 저자 Supp8 (이미 p<5e-8 + LD clump)

원래는 p<5e-8 + LD clumping으로 골라야 하지만, 오프라인이라 저자가 이미 clump한 도구변수(Supplementary Table 8)를 사용.

In [ ]:
s8 = pd.read_csv("Supplementary Table 8.csv", skiprows=1)  # 저자 도구변수
s8[s8.exposure=="FN1"]     # F통계량 = 도구 강도

저자 도구변수 — FN1: 3개 / ALDH2: 15개


SNP,effect_allele,beta.exposure,se,eaf,pval.exposure,F통계량
rs615857,G,-0.0892814,0.014637,0.791706,1.06331E-09,37.20406743
rs10932612,T,-0.21082,0.0119182,0.577616,5.11564E-70,312.8773828
rs17525860,T,-0.136156,0.0147495,0.203103,2.67116E-20,85.21017497


→ 원본 18,851개 → **선택된 3개(FN1)** 만 노출로. **F통계량>10**이면 강한 도구변수(약한 도구 편향 없음).

## STEP 4. 결과(GWAS) 매칭 + harmonise → MR 입력 완성

선택된 도구변수 SNP을 결과 GWAS(GCST, DKD)에서 찾아 붙이고, 대립유전자 방향을 맞춤(harmonise).

In [ ]:
dat = harmonise_data(exposure_dat, outcome_dat)   # 노출+결과 방향 맞춰 합침
dat[dat.exposure=="FN1"]      # 최종 MR 입력

SNP,effect_allele,노출b(발현),결과b(DKD),결과se,결과pval
rs10932612,T,-0.2108,-0.2480,0.103,0.0161
rs17525860,T,-0.1362,-0.0214,0.120,0.8590
rs615857,G,-0.0893,-0.1670,0.121,0.1690


→ 이 표가 **MR의 최종 입력**. SNP마다 '노출b → 결과b' 기울기를 구해 IVW로 합치면 OR (다음: 05_mr_steps).

---
## 요약 — 전처리 흐름
| 단계 | 입력 | 출력 |
| --- | --- | --- |
| 1 원본 | eQTL VCF (18,851 SNP) | 원시 신호 |
| 2 파싱 | FORMAT(ES:SE:LP:AF:SS) 분리 | beta·se·pval·eaf |
| 3 선택 | ∩ Supp8 (p<5e-8+LD 완료) | FN1 3 / ALDH2 14 |
| 4 harmonise | + 결과 GWAS 방향정렬 | MR 입력 표 |

→ 원본을 「파싱 → 도구변수 선택 → 결과 매칭 → 방향정렬」로 가공하면 MR 입력 완성. 그다음 IVW로 인과추정 → FN1 OR 2.78(위험), ALDH2 OR 0.67(보호).

---
# 부록 (선택) — eQTLGen 원본에서 빈도로 beta 직접 만들기

> 위(STEP 1~4)는 저자 **완제품(Supp8 도구변수)** 을 그대로 썼다. 여기서는 저자 완제품 없이 **날것 원본(eQTLGen)** 에서
> 노출을 직접 만들어도 같은 결론이 나오는지 확인. eQTLGen은 beta를 안 주고 **Zscore만** 주므로 **빈도(eaf)로 beta를 역산**해야 함.
> FN1 도구변수 3개로 역산 → 기울기 → IVW → OR 을 하나하나 표로 출력.


In [ ]:
import pandas as pd, numpy as np
# ① eQTLGen 원본(Z, N) + 빈도표(2-4)에서 뽑은 eaf — FN1 3개 SNP (실제값)
ex = pd.DataFrame({
    "SNP":    ["rs10932612","rs17525860","rs615857"],
    "Zscore": [ 17.6889,     -9.23,        6.10   ],   # eQTLGen Zscore
    "N":      [ 30901,        31569,        31684  ],   # 표본수
    "eaf":    [ 0.5776,       0.2031,       0.2083 ],   # 빈도(2-4)
})
print("① 빈도 + 노출(Z) 결합 — eQTLGen은 beta·SE 없이 Zscore만 줌")
ex

① 빈도 + 노출(Z) 결합 — eQTLGen은 beta·SE 없이 Zscore만 줌


,SNP,Zscore,N,eaf
0,rs10932612,17.6889,30901,0.5776
1,rs17525860,-9.2300,31569,0.2031
2,rs615857,6.1000,31684,0.2083


### ② Z → beta·SE 역산 (빈도 eaf 사용)
`beta = Z / √(2·p·(1−p)·(N+Z²))`, `SE = 1 / √(...)`  (p = eaf)

In [ ]:
p=ex["eaf"]; Z=ex["Zscore"]; N=ex["N"]
denom = np.sqrt(2*p*(1-p)*(N + Z**2))
ex["beta_발현"] = (Z/denom).round(4)     # 노출 = SNP이 FN1 발현 얼마나 바꾸나
ex["SE_발현"]   = (1/denom).round(4)
print("② 역산 완료 → 이제 beta·SE 생김 = 노출(도구변수) 완성")
ex

② 역산 완료 → 이제 beta·SE 생김 = 노출(도구변수) 완성


,SNP,Zscore,N,eaf,beta_발현,SE_발현
0,rs10932612,17.6889,30901,0.5776,0.1433,0.0081
1,rs17525860,-9.2300,31569,0.2031,-0.0912,0.0099
2,rs615857,6.1000,31684,0.2083,0.0596,0.0098


### ③ 결과(DKD GWAS) 붙이기 + SNP별 기울기
기울기 = beta_DKD ÷ beta_발현  ("발현 1 올릴 때 DKD 얼마나")

In [ ]:
# 결과(GCST GWAS)에서 같은 SNP의 beta_DKD·SE (harmonise 방향 정렬 후)
ex["beta_DKD"] = [ 0.248, -0.021, 0.167 ]      # SNP→DKD 로그오즈
ex["SE_DKD"]   = [ 0.103,  0.120, 0.121 ]
ex["기울기"]   = (ex["beta_DKD"]/ex["beta_발현"]).round(3)   # 인과 크기(발현→DKD)
print("③ SNP마다 기울기 = beta_DKD ÷ beta_발현")
ex[["SNP","beta_발현","beta_DKD","기울기"]]

③ SNP마다 기울기 = beta_DKD ÷ beta_발현


,SNP,beta_발현,beta_DKD,기울기
0,rs10932612,0.1433,0.248,1.731
1,rs17525860,-0.0912,-0.021,0.230
2,rs615857,0.0596,0.167,2.802


### ④ IVW로 합치기 (1/SE² 가중평균) → OR
기울기가 SNP마다 다름 → 정밀한(오차 작은) SNP에 힘을 더 줘 하나로 합침.

In [ ]:
se_slope = np.abs(ex["기울기"]) * np.sqrt((ex["SE_DKD"]/ex["beta_DKD"])**2 + (ex["SE_발현"]/ex["beta_발현"])**2)
w = 1/se_slope**2                       # 가중치 = 1/SE² (정밀한 SNP에 힘↑)
beta_ivw = np.sum(w*ex["기울기"])/np.sum(w)
se_ivw   = 1/np.sqrt(np.sum(w))
OR = np.exp(beta_ivw)
from math import erf, sqrt
z = beta_ivw/se_ivw
pval = 2*(1-0.5*(1+erf(abs(z)/sqrt(2))))
print(f"④ IVW 합친 beta = {beta_ivw:.3f}  →  OR = exp(beta) = {OR:.2f}  (p ≈ {pval:.4f})")
print("   → OR > 1 = FN1 발현↑ → DKD 위험↑ (인과, 유의)")
pd.DataFrame({"SNP":ex["SNP"],"기울기":ex["기울기"],"SE_기울기":se_slope.round(3),"가중치(1/SE²)":w.round(2)})

④ IVW 합친 beta = 1.502  →  OR = exp(beta) = 4.49  (p ≈ 0.0135)
   → OR > 1 = FN1 발현↑ → DKD 위험↑ (인과, 유의)


,SNP,기울기,SE_기울기,가중치(1/SE²)
0,rs10932612,1.731,0.726,1.90
1,rs17525860,0.230,1.315,0.58
2,rs615857,2.802,2.082,0.23


→ **결론**: FN1 3개 SNP 기울기(1.73·0.23·2.78)를 IVW로 합쳐 **OR ≈ 4.5** (eQTLGen 버전). 방향(위험)·유의성 재현.
(※ 주 재현은 저자 VCF 노출로 OR 2.78 — 노출 단위 차이. 방향·유의는 동일)